In [3]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.9.1+cu128
CUDA available: True


In [4]:
!pip install huggingface_hub -q

import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

from huggingface_hub import snapshot_download

# Download Qwen2.5-7B-Instruct
snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir="/workspace/models/qwen2.5-7b-instruct"
)

# Download AV checkpoint
snapshot_download(
    repo_id="kitft/nla-qwen2.5-7b-L20-av",
    local_dir="/workspace/models/nla-av"
)

# Download AR checkpoint
snapshot_download(
    repo_id="kitft/nla-qwen2.5-7b-L20-ar",
    local_dir="/workspace/models/nla-ar"
)

print("✅ Download ทั้งหมดเสร็จแล้ว!")

✅ Download ทั้งหมดเสร็จแล้ว!


In [5]:
import os

for path in [
    "/workspace/models/qwen2.5-7b-instruct",
    "/workspace/models/nla-av",
    "/workspace/models/nla-ar"
]:
    files = os.listdir(path)
    print(f"✅ {path.split('/')[-1]}: {len(files)} files")
    print(f"   {files}\n")

✅ qwen2.5-7b-instruct: 15 files
   ['vocab.json', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors.index.json', 'model-00004-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00001-of-00004.safetensors', 'merges.txt', 'generation_config.json', 'config.json', 'README.md', 'LICENSE', '.gitattributes', '.cache']

✅ nla-av: 19 files
   ['vocab.json', 'tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json', 'nla_meta.yaml', 'model.safetensors.index.json', 'model-00004-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00001-of-00004.safetensors', 'merges.txt', 'generation_config.json', 'config.json', 'chat_template.jinja', 'added_tokens.json', 'README.md', 'LICENSE', '.gitattributes', '.cache']

✅ nla-ar: 19 files
   ['model-00003-of-00003.safetensors', 'value_head.safetensors', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'nla_meta.yaml', 'model.safet

In [6]:
!pip install accelerate -q

In [7]:
!nvidia-smi

Thu Jun 18 14:20:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:56:00.0 Off |                    0 |
|  0%   33C    P8             35W /  300W |       3MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!nvidia-smi | grep MiB

|  0%   32C    P0             35W /  300W |       3MiB /  46068MiB |      0%      Default |


In [9]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
x = torch.ones(3,3).cuda()
print('Simple CUDA tensor:', x)
print('✅ CUDA ทำงานได้!')

PyTorch: 2.9.1+cu128
CUDA available: True
Simple CUDA tensor: tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
✅ CUDA ทำงานได้!


In [10]:
import torch

# ทดสอบ allocate VRAM ขนาดใหญ่ ~15GB
try:
    x = torch.zeros(15 * 1024 * 1024 * 1024 // 2, dtype=torch.bfloat16).cuda()
    print(f"✅ Allocate 15GB VRAM สำเร็จ!")
    del x
    torch.cuda.empty_cache()
except Exception as e:
    print(f"❌ Error: {e}")

✅ Allocate 15GB VRAM สำเร็จ!


In [11]:
# สร้าง script
script = """
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = '/workspace/models/qwen2.5-7b-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map='auto'
)
model.eval()
print('✅ สำเร็จ!')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
"""

with open('/workspace/load_qwen.py', 'w') as f:
    f.write(script)
print("✅ สร้างไฟล์แล้ว!")

✅ สร้างไฟล์แล้ว!


In [12]:
!pip install "sglang[all]==0.5.6" -q
!pip install accelerate -q
!cd /workspace/natural_language_autoencoders && pip install -e . -q

In [13]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
!ls /workspace/models/

PyTorch: 2.9.1+cu128
CUDA: True
GPU: NVIDIA A40
nla-ar	nla-av	qwen2.5-7b-instruct


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/workspace/models/qwen2.5-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
print("✅ โหลด Qwen สำเร็จ!")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ โหลด Qwen สำเร็จ!
VRAM: 15.2 GB


In [17]:
import torch
import numpy as np

# เคลียร์ hook เก่าทั้งหมดก่อน
for layer in model.model.layers:
    layer._forward_hooks.clear()

activation = {}

def hook_fn(module, input, output):
    # output เป็น tensor โดยตรง (batch, seq, hidden)
    if isinstance(output, tuple):
        hidden = output[0]
    else:
        hidden = output
    hidden = hidden[:, -1, :].float()  # last token
    hidden = hidden / hidden.norm(dim=-1, keepdim=True)  # L2-normalize
    activation["layer20"] = hidden.detach().cpu()

hook = model.model.layers[20].register_forward_hook(hook_fn)

text = "The capital of France is Paris."
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    model(**inputs)

hook.remove()

print("✅ shape:", activation["layer20"].shape)
print("✅ norm:", activation["layer20"].norm().item())
print("✅ dtype:", activation["layer20"].dtype)

✅ shape: torch.Size([1, 3584])
✅ norm: 0.9999998807907104
✅ dtype: torch.float32


In [18]:
import gc

del model
torch.cuda.empty_cache()
gc.collect()

print(f"VRAM freed: {torch.cuda.memory_allocated()/1e9:.1f} GB")

VRAM freed: 15.2 GB


In [19]:
!nvidia-smi | grep MiB

|  0%   46C    P0            117W /  300W |   14881MiB /  46068MiB |      0%      Default |
|    0   N/A  N/A            8196      C   /usr/local/bin/python                 14872MiB |


In [2]:
import sys
sys.path.insert(0, '/workspace/natural_language_autoencoders')

import nla
print(dir(nla))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [3]:
import os

# ดูไฟล์ทั้งหมดใน nla package
for root, dirs, files in os.walk('/workspace/natural_language_autoencoders/nla'):
    for f in files:
        print(os.path.join(root, f))

/workspace/natural_language_autoencoders/nla/train_actor.py
/workspace/natural_language_autoencoders/nla/tis_metrics.py
/workspace/natural_language_autoencoders/nla/storage.py
/workspace/natural_language_autoencoders/nla/schema.py
/workspace/natural_language_autoencoders/nla/reward.py
/workspace/natural_language_autoencoders/nla/models.py
/workspace/natural_language_autoencoders/nla/loss.py
/workspace/natural_language_autoencoders/nla/injection.py
/workspace/natural_language_autoencoders/nla/embed_store.py
/workspace/natural_language_autoencoders/nla/data_source.py
/workspace/natural_language_autoencoders/nla/config.py
/workspace/natural_language_autoencoders/nla/arch_adapters.py
/workspace/natural_language_autoencoders/nla/_sglang_sft_stubs.py
/workspace/natural_language_autoencoders/nla/__init__.py
/workspace/natural_language_autoencoders/nla/__pycache__/__init__.cpython-312.pyc
/workspace/natural_language_autoencoders/nla/scripts/rl_preflight.py
/workspace/natural_language_autoencod

In [4]:
# เช็ค injection.py — น่าจะเป็นไฟล์หลักสำหรับ inference
!cat /workspace/natural_language_autoencoders/nla/injection.py

"""Pure injection-hook logic — extracted for testability.

The most correctness-critical path in NLA: if injection fails or hits the wrong
position, the model sees the literal ㊗ character and outputs Chinese. This
function is the one place that must be right, so it's pure and unit-testable.
"""

import torch


def inject_at_marked_positions(
    input_ids: torch.Tensor,
    embeddings: torch.Tensor,
    vectors: torch.Tensor,
    inj_id: int,
    left_id: int,
    right_id: int,
    seq_slice: tuple[int, int] | None = None,
) -> torch.Tensor:
    """Overwrite embedding rows at injection-marker positions with activation vectors.

    input_ids: [B, S] — or [1, T_packed] for thd layout. The FULL token stream
        (broadcast across TP ranks — identical everywhere).
    embeddings: [B, S, d] (unsharded) or [B, S_local, d] (seq_slice set). The
        embedding layer output. Cloned; original unchanged.
    vectors: [N, d] — activation vectors in microbatch order. N = number of
        in

In [5]:
!ls /workspace/natural_language_autoencoders/docs/
!cat /workspace/natural_language_autoencoders/docs/inference.md

design.md  inference.md  setup.md
# NLA Inference

Standalone inference client + recipe for NLA (Natural Language Autoencoder) models.

An **NLA pair** is two fine-tuned LMs that together map activation vectors to
natural language and back:

| | direction | mechanism |
|---|---|---|
| **AV** (Activation Verbaliser) | `vector → text` | inject vector as a 1-token embedding into a fixed prompt, autoregress |
| **AR** (Activation Reconstructor) | `text → vector` | truncated K+1-layer LM + Linear(d,d) head, extract at final token |

The round-trip **MSE(reconstructed, original)** measures how well the
verbalization captured the vector's content — it was the RL reward signal
during AV training. Low MSE ⟹ the AR can recover the original
direction from the AV's words alone.

**What's here:**
- `nla_inference.py` — single-file AV client (no heavy deps, SGLang input_embeds)
- `examples/` — worked transcripts with per-token MSE
- This README — full recipe, model-specific params, AR architecture, 

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# โหลด Qwen
model_path = "/workspace/models/qwen2.5-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
print(f"✅ โหลด Qwen สำเร็จ! VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# โหลด HaluEval dataset
dataset = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
samples = dataset.select(range(200))
print(f"✅ Dataset: {len(samples)} samples")

# Extract Layer 20 activations
activations = []
labels = []

def hook_fn(module, input, output):
    hidden = output[0] if isinstance(output, tuple) else output
    hidden = hidden[:, -1, :].float()
    hidden = hidden / hidden.norm(dim=-1, keepdim=True)
    activations.append(hidden.detach().cpu())

hook = model.model.layers[20].register_forward_hook(hook_fn)

for i, sample in enumerate(samples):
    prompt = f"Question: {sample['question']}\nAnswer: {sample['answer']}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    
    with torch.no_grad():
        model(**inputs)
    
    labels.append(1 if sample["hallucination"] == "yes" else 0)
    
    if (i+1) % 20 == 0:
        print(f"Progress: {i+1}/200")

hook.remove()

# Save
activations_np = torch.cat(activations, dim=0).numpy()
labels_np = np.array(labels)

np.save("/workspace/activations_L20.npy", activations_np)
np.save("/workspace/labels_L20.npy", labels_np)

print(f"✅ Saved! activations: {activations_np.shape}, labels: {labels_np.shape}")
print(f"Hallucinated: {labels_np.sum()}, Truthful: {(labels_np==0).sum()}")

: 

In [2]:
import gc
del model
torch.cuda.empty_cache()
gc.collect()
print("✅ Qwen unloaded!")

✅ Qwen unloaded!


In [4]:
!nvidia-smi | grep MiB

|  0%   47C    P0            117W /  300W |     353MiB /  46068MiB |      0%      Default |
|    0   N/A  N/A            9942      C   /usr/local/bin/python                   344MiB |


In [5]:
!find /workspace/natural_language_autoencoders -name "nla_inference.py"

/workspace/natural_language_autoencoders/nla_inference.py


In [6]:
!cat /workspace/natural_language_autoencoders/nla_inference.py

"""NLA actor inference via SGLang input_embeds — single-file, no nla package deps.

An NLA (Natural Language Autoencoder) pair is two fine-tuned LMs that together
map activation vectors to natural language and back:

  ACTOR  (activation verbalizer)  : hidden-state vector  →  text
                                    [inject vector as a 1-token embedding
                                     into a fixed prompt, then autoregress]

  CRITIC (activation reconstructor): text  →  hidden-state vector
                                    [truncated K+1-layer LM + Linear(d,d)
                                     head, extract at final token]

The round-trip — extract → ACTOR verbalizes → CRITIC reconstructs → MSE against
original — measures how well the verbalization captured the vector's content.
That MSE was the RL reward signal during actor training: low MSE means the
critic can recover the original direction from the actor's words alone.

This file contains both halves:
  NLAClient  — actor 

In [7]:
import sys
sys.path.insert(0, '/workspace/natural_language_autoencoders')

import numpy as np
import torch
from nla_inference import NLAClient, NLACritic

# โหลด activations ที่ save ไว้
activations = np.load("/workspace/activations_L20.npy")
labels = np.load("/workspace/labels_L20.npy")
print(f"✅ Loaded: {activations.shape}, labels: {labels.shape}")

# Init NLAClient (AV via SGLang)
client = NLAClient(
    checkpoint_dir="/workspace/models/nla-av",
    sglang_url="http://127.0.0.1:30000"
)

# Init NLACritic (AR บน CPU เพื่อประหยัด VRAM)
critic = NLACritic(
    checkpoint_dir="/workspace/models/nla-ar",
    device="cpu",
    dtype=torch.bfloat16
)

# ทดสอบ 1 sample ก่อน
v = activations[0]
explanation = client.generate(v)
print(f"\n✅ Explanation:\n{explanation}")

mse, cos = critic.score(explanation, v)
print(f"\n✅ MSE: {mse:.4f}")
print(f"✅ Cosine: {cos:.4f}")

✅ Loaded: (200, 3584), labels: (200,)
[NLAClient] nla-av: d_model=3584 inj_scale=150.0 embed_scale=1.00 inj_char='㈎'(id=149705)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at /workspace/models/nla-ar and are newly initialized: ['lm_head.weight', 'model.norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[NLACritic] 21 layers  d_model=3584  mse_scale=59.87

✅ Explanation:
Structured informational format with a factual description pattern, presenting details about a Disney film titled "Carmen" as a Christian concept.

The sentence "The earliest known founding date of 'Superman' as a religious work is December 1st. " signals an answer is being delivered, establishing a comparative context implying the film's origin date before the term "Holy Day."

Final token "work. " closes a clause mid-sentence ("The earliest known founding date and type of Christmas. Superman's creation is a Non-Fiction film. "), strongly expecting "It was founded in..." or "The Pixar's Creation Date is..." or "This work was determined to be before..."

✅ MSE: 0.4284
✅ Cosine: 0.7858


In [8]:
from tqdm import tqdm
import numpy as np

all_explanations = []

for i in tqdm(range(len(activations))):
    v = activations[i]
    explanation = client.generate(v)
    all_explanations.append(explanation)

# Save
np.save("/workspace/explanations.npy", np.array(all_explanations))
print(f"✅ Saved {len(all_explanations)} explanations!")

100%|██████████| 200/200 [15:48<00:00,  4.74s/it]

✅ Saved 200 explanations!


In [11]:
import torch
import numpy as np
import sys
sys.path.insert(0, '/workspace/natural_language_autoencoders')
from nla_inference import NLACritic

# โหลด data
activations = np.load("/workspace/activations_L20.npy")
explanations = np.load("/workspace/explanations.npy", allow_pickle=True)
labels = np.load("/workspace/labels_L20.npy")
print(f"✅ Loaded: {activations.shape}, {len(explanations)} explanations")

# Load AR บน CUDA
critic = NLACritic(
    checkpoint_dir="/workspace/models/nla-ar",
    device="cuda",
    dtype=torch.bfloat16
)

# Reconstruct ทุก sample
from tqdm import tqdm
pred_vecs = []

for exp in tqdm(explanations):
    pred = critic.reconstruct(exp)
    pred_vecs.append(pred)

pred_vecs = torch.stack(pred_vecs)  # [200, 3584]

# คำนวณ fve_nrm ตาม doc
mse_scale = critic.mse_scale
gold = torch.tensor(activations, dtype=torch.float32)
gold_n = gold / gold.norm(dim=-1, keepdim=True) * mse_scale
pred_n = pred_vecs / pred_vecs.norm(dim=-1, keepdim=True) * mse_scale

mu = gold_n.mean(dim=0)  # ห้าม normalize!
fve = 1 - ((pred_n - gold_n)**2).mean() / ((gold_n - mu)**2).mean()

# cos similarity
cos_sim = torch.nn.functional.cosine_similarity(pred_n, gold_n, dim=-1)

print(f"\n✅ fve_nrm:     {fve.item():.4f}  (target: ~0.752)")
print(f"✅ Mean cos:    {cos_sim.mean().item():.4f}  (target: ~0.90)")
print(f"✅ Mean MSE:    {(2*(1-cos_sim)).mean().item():.4f}  (target: ~0.20)")

✅ Loaded: (200, 3584), 200 explanations


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at /workspace/models/nla-ar and are newly initialized: ['lm_head.weight', 'model.norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[NLACritic] 21 layers  d_model=3584  mse_scale=59.87


100%|██████████| 200/200 [00:06<00:00, 29.11it/s]


✅ fve_nrm:     0.2886  (target: ~0.752)
✅ Mean cos:    0.8030  (target: ~0.90)
✅ Mean MSE:    0.3940  (target: ~0.20)


In [10]:
# Debug: เช็คว่าคำนวณถูกไหม
print(f"gold_n norm mean: {gold_n.norm(dim=-1).mean():.2f}")  # ควรเป็น ~59.87
print(f"pred_n norm mean: {pred_n.norm(dim=-1).mean():.2f}")  # ควรเป็น ~59.87
print(f"variance gold: {((gold_n - mu)**2).mean():.4f}")
print(f"variance pred-gold: {((pred_n - gold_n)**2).mean():.4f}")

gold_n norm mean: 59.87
pred_n norm mean: 59.87
variance gold: 0.5539
variance pred-gold: 0.3940
